# 02 · Preprocessing

**What this notebook does.** It documents and (only if asked) runs the preprocessing that turns every raw Neuropixels
recording into the **1000 Hz, un-referenced LFP** used by all analyses, and it lists the **bad channels** removed on the way.

**Run order in the project:** 01 → **02** → 03 → 04 → 05 → 06 → 07. Needs the tables of 01 (`usable_sessions.csv`, `final_sessions.csv`).
The preprocessing itself was run once (2026-10-16, 24.6 h for 63 sessions); it does **not** need to be run again.

**Steps**

| Step | What it does | Output |
|---|---|---|
| 1 | Preprocessing chain for every usable session (only with `RUN = True`) | 1000 Hz recordings in `2_processed\<session>\amir_data\down_sampled\1000_Hz` |
| 2 | Bad channels per session, read back from the saved recordings | `preprocessing_log.csv` |
| 3 | How often each channel is bad, per rat + figure | `bad_channel_frequency.csv`, `bad_channels_per_rat.png` |
| 4 | All excluded channels per session (preprocessing + found later in 04/05) | `excluded_channels_per_session.csv` |
| 5 | Example of a removed channel next to its neighbours | figure |

**How to use it:** run the settings cell first; then any cell. Nothing is written unless `SAVE = True`
(and the recordings are only processed with `RUN = True`). Change things only in the settings cell.
Functions: `fx_preprocessing.py`; colours and saving: `fx_common.py`.

## Settings

**Paths:** `CODE_DIR` (notebooks + `fx_*.py`), `DATA_ROOT` with `RAW` (raw `ap.bin`) and `PROC` (processed sessions), `AMIR`
(your data folder), `TABLES` (session and trial tables from 01; **temporarily the old folder in the repository**),
`FIG_DIR` (figures of this notebook), `QC_DIR` (the example figure of step 5, kept with the QC figures).

**Saving:** `SAVE = False` writes nothing. `True` writes figures to `FIG_DIR` and tables to `TABLES`.
To save one cell elsewhere: add `folder=r"...", force=True` to its save line.

**Preprocessing (step 1):** `RUN = False` only reports which sessions are done. `True` processes the sessions that have no
`DONE.txt` yet. `N_JOBS = 4` and `CHUNK = "5s"` (with 22 jobs and 10 s chunks the computer ran out of memory).
`PILOT` is the old pilot session, preprocessed earlier by hand with the same chain (it has no `DONE.txt`).

**Example (step 5):** `EX_SESSION`, `EX_CHANNEL`: the session and the removed channel to show (e.g. rat 134: AP27, rat 135: AP132).

In [ ]:
# ══ SETTINGS — run first ══
import os, sys, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

# ── paths ──
CODE_DIR  = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"                  # folder with the notebooks + fx_*.py
DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"             # project data on the server
RAW       = os.path.join(DATA_ROOT, "0_raw")                                       # raw recordings (ap.bin)
PROC      = os.path.join(DATA_ROOT, "2_processed")                                 # 1000 Hz recordings + share/trials.csv
AMIR      = os.path.join(DATA_ROOT, "amir_data")                                   # your own data folder (all outputs)
TABLES    = os.path.join(CODE_DIR, "tables")                                       # TEMPORARY: old place; later os.path.join(AMIR, "tables")
FIG_DIR   = os.path.join(AMIR, "figures", "02_preprocessing")                      # figures of this notebook (if SAVE)
QC_DIR    = os.path.join(AMIR, "QC")                                               # example figure of step 5 (if SAVE)

# ── saving ──
SAVE = False                                                                       # True = write figures and tables

# ── preprocessing (step 1) ──
RUN    = False                                                                     # True = process sessions without DONE.txt
N_JOBS = 4                                                                         # parallel jobs (more ran out of memory)
CHUNK  = "5s"                                                                      # chunk length per job
PILOT  = "20260522_r0000136_wt_1750"                                               # old pilot, preprocessed by hand (no DONE.txt)

# ── example (step 5) ──
EX_SESSION, EX_CHANNEL = "20260529_r0000136_wt_1318", "AP138"                      # rat 136: its only removed channel

# ── code + tables ──
sys.path.insert(0, CODE_DIR)
import fx_common as fxc, fx_preprocessing as fp
importlib.reload(fxc); importlib.reload(fp)
read = lambda name: pd.read_csv(os.path.join(TABLES, name + ".csv"), dtype={"rat": str})
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
fxc.check_paths(RAW=RAW, PROC=PROC, TABLES=TABLES)
fxc.set_saving(SAVE, fig_dir=FIG_DIR, table_dir=TABLES)
usable, final = read("usable_sessions"), read("final_sessions")
print(f"usable (preprocessed): {len(usable)} | final (analysed): {len(final)}")

## Step 1 · Preprocessing chain (run once, 2026-10-16)

**What it does, per session** (SpikeInterface 0.105.0):
1. reads the raw `ap.bin` (30 kHz, 384 channels);
2. **band-pass 1–300 Hz** (LFP range; float32);
3. **`detect_bad_channels`** (SpikeInterface default method) and **removes** those channels;
4. **`phase_shift`**: corrects the small time offset between channels caused by the probe's multiplexed sampling;
5. **notch 50 Hz** (q = 30): removes mains noise;
6. **resample to 1000 Hz**;
7. saves the result **without any re-referencing** to `2_processed\<session>\amir_data\down_sampled\1000_Hz` and writes `DONE.txt` at the end.

Re-referencing (CMR or local) is done later, at loading time (06), so all references start from the same data.
A half-finished folder (after a crash) is deleted and redone. The old pilot was made earlier by hand with the same chain
(SpikeInterface 0.104.8, 1 s chunks).

**With `RUN = False`** the cell only lists the sessions still to do. **Expected:** 63 done, 0 to do.
**With `RUN = True`** run and error logs are appended to `preprocessing_run_log.csv` / `preprocessing_errors.csv` in `TABLES`.

In [ ]:
todo = fp.preprocess_all(usable, RAW, PROC, PILOT, run=RUN, n_jobs=N_JOBS, chunk=CHUNK,
                         run_log=os.path.join(TABLES, "preprocessing_run_log.csv"), err_log=os.path.join(TABLES, "preprocessing_errors.csv"))

## Step 2 · Bad channels per session

**What it does.** The original preprocessing log was lost, so it is rebuilt from the saved recordings: for every usable session,
the bad channels = all 384 channels (AP0–AP383) **minus** the channels present in the saved 1000 Hz recording
(= the channels `detect_bad_channels` removed). Only the metadata is read. Check: kept + bad = 384 in every session.

**Expected (2026-10-16):** rat 134: 7–9 bad channels, rat 135: 6–8, rat 136: 1 (AP138).

**Input:** `usable_sessions.csv` + the recordings. **Saves:** `preprocessing_log.csv` (session, rat, n_bad, bad_ids, n_channels_kept).

In [ ]:
prep_log = fp.bad_channel_log(usable, PROC)
fxc.save_table(prep_log, "preprocessing_log")

## Step 3 · Bad channels per rat

**What it does.** Using the preprocessing log of the 60 final sessions:
- for each rat and channel, in how many sessions (and what %) the channel was removed;
- which bad-channel **lists** occur and in how many sessions (identical lists = the same broken channels in every session).

**Figure:** top = % of the rat's sessions in which each channel is bad (number of sessions above the bar);
bottom = session × channel (black = removed in that session).

**Expected (2026-10-16):** rat 134 core AP27, AP64, AP89, AP179, AP187, AP303, AP332 (+AP63, AP90 in some sessions);
rat 135 core AP132, AP193, AP322, AP325, AP329, AP376 (+AP326, AP362 in some); rat 136 only AP138.

**Input:** `preprocessing_log.csv`, `final_sessions.csv`. **Saves:** `bad_channel_frequency.csv`, figure `bad_channels_per_rat.png`.

In [ ]:
freq, fig = fp.bad_channel_frequency(read("preprocessing_log"), final)
fxc.save_table(freq, "bad_channel_frequency"); fxc.save_fig(fig, "bad_channels_per_rat", dpi=150); plt.show()

## Step 4 · All excluded channels per session

**What it does.** Joins two lists per final session:
1. **preprocessing:** channels removed by `detect_bad_channels` (step 2);
2. **found later:** `extra_bad_channels.csv` (made in 04 QC and 05 event check: channels with abnormal signal or ERPs, plus
   per-rat hardware channels removed in every session of that rat: AP326 + AP362 in rat 135, AP63 + AP90 in rat 134).

Total = the union (a channel in both lists counts once); channels used = 384 − total. It also counts how many later-found
channels were already removed in preprocessing, and lists sessions in `extra_bad_channels.csv` that are not final sessions.

**Input:** `preprocessing_log.csv`, `final_sessions.csv`, `extra_bad_channels.csv`. **Saves:** `excluded_channels_per_session.csv`.

In [ ]:
excluded = fp.excluded_channels(read("preprocessing_log"), final, pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv")))
fxc.save_table(excluded, "excluded_channels_per_session")

## Step 5 · Example of a channel removed by `detect_bad_channels`

**What it does.** Removed channels are missing from the 1000 Hz files, so the **raw `ap.bin`** is used: band-pass 1–300 Hz
and resampling to 1000 Hz as in preprocessing (no bad-channel removal, phase shift or notch). It shows 2 s from the middle of the
recording for the removed channel (red) and its 4 nearest **good** neighbours on the same shank (within ±75 µm depth; other
removed channels are left out), sorted by depth and labelled with their depth (µm from the tip).

Note: the old description also mentioned a power spectrum, but the code only draws the raw traces.

**Saves:** figure `detect_bad_channels example <channel> <session>.png` in `QC_DIR`.

In [ ]:
removed = set(read("preprocessing_log").set_index("session").loc[EX_SESSION, "bad_ids"].split(";"))
fig = fp.example_bad_channel(RAW, EX_SESSION, EX_CHANNEL, removed)
fxc.save_fig(fig, f"detect_bad_channels example {EX_CHANNEL} {EX_SESSION}", folder=QC_DIR); plt.show()